# ДЗ 7. Срезы orders.csv: groupby, pivot_table, merge
Срок: **до воскресенья 29.11, 24:00**.
Сдача: папка `hw07` репозитория `dsa-homework` (`hw07.ipynb`): `git add hw07` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать. Рядом с каждой долей или средним указывайте, на скольких заказах оно посчитано.

Инструменты: всё с уроков 1–7. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
t = pd.DataFrame({"c": ["a", "b", "a"], "x": [1, 2, 3]})
print(t.groupby("c")["x"].sum().tolist())
# 2
t = pd.DataFrame({"c": ["a", "a", "b"], "d": [1, None, 3]})
print(t.groupby("c")["d"].count().tolist(), t.groupby("c").size().tolist())
# 3
t = pd.DataFrame({"c": ["a", "a", "b"], "m": [9, 9, 10], "x": [10, 30, 5]})
print(t.pivot_table(index="c", columns="m", values="x").loc["a", 9])
# 4
a = pd.DataFrame({"id": [1, 2, 3]})
b = pd.DataFrame({"id": [1, 2], "ch": ["x", "y"]})
print(len(a.merge(b, on="id")), len(a.merge(b, on="id", how="left")))
# 5
b = pd.DataFrame({"id": [1, 1, 2], "ch": ["x", "x", "y"]})
print(len(a.merge(b, on="id", how="left")))
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [ ]:
1. [1, 3] groupby("c") объединяет две строки с a сумма x для них равна 1+2=3

In [ ]:
2. [1, 1] и [2, 1] count не считает none а size считает все строки

In [ ]:
3. 20 после фильтра остаются строки с индексами a и b но loc["a", 9] возвращает значение x для города a и месяца 9

In [ ]:
4. 2 и 3 при inner остаются только id 1 и 2 при left  все три строки таблицы a

In [ ]:
5. 5 В таблице b два совпадения для id=1 поэтому эта строка таблицы a повторится дважды остальные строки дадут ещё два результата

In [1]:
import pandas as pd

In [2]:
t = pd.DataFrame({"c": ["a", "b", "a"], "x": [1, 2, 3]})
print(t.groupby("c")["x"].sum().tolist())

[4, 2]


In [3]:
t = pd.DataFrame({"c": ["a", "a", "b"], "d": [1, None, 3]})
print(t.groupby("c")["d"].count().tolist(), t.groupby("c").size().tolist())

[1, 1] [2, 1]


In [4]:
t = pd.DataFrame({"c": ["a", "a", "b"], "m": [9, 9, 10], "x": [10, 30, 5]})
print(t.pivot_table(index="c", columns="m", values="x").loc["a", 9])

20.0


In [5]:
a = pd.DataFrame({"id": [1, 2, 3]})
b = pd.DataFrame({"id": [1, 2], "ch": ["x", "y"]})
print(len(a.merge(b, on="id")), len(a.merge(b, on="id", how="left")))

2 3


In [6]:
b = pd.DataFrame({"id": [1, 1, 2], "ch": ["x", "x", "y"]})
print(len(a.merge(b, on="id", how="left")))

4


In [7]:
# Загрузка orders.csv и customers.csv: в Colab нажмите «Выбрать файлы» и укажите оба файла из LMS.
# Если файлы уже лежат рядом с ноутбуком, загрузка пропускается.
import os
if not (os.path.exists("orders.csv") and os.path.exists("customers.csv")):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv и customers.csv в одну папку с ноутбуком")
print("orders.csv:", os.path.exists("orders.csv"), "| customers.csv:", os.path.exists("customers.csv"))

orders.csv: True | customers.csv: True


In [8]:
# Очистка урока 6 одной ячейкой: суммы, города, дубли, даты, TEST
import pandas as pd
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

df = pd.read_csv("orders.csv")
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
df["city"] = df["city"].str.strip().str.lower().map(CITY)
df = df.drop_duplicates()
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.month
df = df[df["customer_id"] != "TEST"].copy()    # copy: дальше добавим столбцы
df.shape

(5280, 9)

In [9]:
df["cancel"] = df["status"] == "отменён"
paid = df[df["status"] == "оплачен"]
cu = pd.read_csv("customers.csv")
len(paid), cu.shape

(4739, (2200, 4))

Можно вместо ячейки очистки взять свою функцию `clean()` из ДЗ 6. Проверка: 5 280 строк, 4 739 оплаченных.

## [База] 1. Сводка по городам одной таблицей
Через `agg` с именами: число заказов, доля отмен, средний срок доставки по городам. Добавьте столбец «выручка оплаченных»: `by_city["revenue"] = paid.groupby("city")["amount"].sum()` (Pandas выровняет по ярлыкам, урок 5). Отсортируйте по выручке.
Какой город выделяется сразу по двум показателям?

In [13]:
# 1
by_city = df.groupby("city").agg(
    orders=("order_id", "count"),
    cancel_share=("cancel", "mean"),
    avg_days=("delivery_days", "mean")
)

In [14]:
by_city["revenue"] = paid.groupby("city")["amount"].sum()

In [15]:
by_city.sort_values("revenue", ascending=False).round(2)

,orders,cancel_share,avg_days,revenue
city,,,,
Алматы,2116,0.08,2.00,29144090.0
Астана,1339,0.09,2.02,17884750.0
Шымкент,776,0.21,4.45,8602660.0
Караганда,519,0.09,1.94,7020200.0
Актобе,387,0.11,2.07,5117380.0
Атырау,143,0.08,2.04,1893850.0


In [ ]:
шымкент выделяется сразу по двум показателям это доля отмен около 20.6%, средний срок доставки 4.45 дня 
Доля рассчитана по 776 заказам шымкента

## [База] 2. ДЗ 4 одной строкой
В ДЗ 4 вы считали категории циклами и NumPy. Повторите через `pivot_table`: выручка оплаченных по категориям и месяцам, изменение в %, средний чек по категориям и месяцам.
Сверьте с ответами ДЗ 4: лидер Электроника 34,4 млн ₸, Одежда −7,2%, средний чек Электроники в октябре 33 600 ₸. Сколько строк кода было и сколько стало?

In [16]:
category_revenue = paid.pivot_table(
    index="category", columns="month", values="amount", aggfunc="sum"
)

In [17]:
category_revenue

month,9,10
category,,
Дом,4997730.0,5125790.0
Книги,1065300.0,1140910.0
Красота,2610950.0,2678990.0
Одежда,9167100.0,8505730.0
Электроника,17402330.0,16968100.0


In [18]:
((category_revenue[10] / category_revenue[9] - 1) * 100).round(1)

category
Дом            2.6
Книги          7.1
Красота        2.6
Одежда        -7.2
Электроника   -2.5
dtype: float64

In [19]:
paid.pivot_table(
    index="category", columns="month", values="amount", aggfunc="mean"
).round(2)

month,9,10
category,,
Дом,10499.43,10768.47
Книги,4020.00,3815.75
Красота,6187.09,6158.60
Одежда,13520.80,13045.60
Электроника,32772.75,33600.20


In [ ]:
по расчёту дз4 лидирует электроника с выручкой 34.37млн за сентябрь и октябрь
выручка одежды снизилась примерно на 7.2% к сентябрю
средний чек электроники в октябре составил 33600млн
для этой задачи 5 строк кода дз4 создававших categories, маску, суммы по месяцам и table, заменяются одним вызовом pivot_table

## [База] 3. Отмены и срок доставки
По `delivery_days`: число заказов и доля отмен. Во сколько раз отменяют чаще при доставке за 8 дней, чем за 1 день?
Что это значит для Шымкента, где в октябре доставка 5–8 дней?

In [23]:
by_days = df.groupby("delivery_days").agg(
    orders=("order_id", "count"),
    cancel_share=("cancel", "mean")
)

In [24]:
by_days.round(3)

,orders,cancel_share
delivery_days,,
1.0,1557,0.042
2.0,1570,0.080
3.0,1590,0.134
5.0,86,0.221
6.0,120,0.217
7.0,95,0.347
8.0,104,0.433


In [25]:
round(by_days.loc[8, "cancel_share"] / by_days.loc[1, "cancel_share"], 2)

np.float64(10.21)

In [ ]:
при более долгой доставке доля отмен выше это особенно важно для шымкента где в октябре встречались сроки 5–8 дней
Эти данные показывают связь срока доставки с отменами но сами по себе не доказывают что именно задержка стала причиной каждой отмены

## [Стандарт] 4. Исправьте код от AI
AI посчитал изменение выручки по городам и пишет менеджеру: «В Шымкенте выручка выросла на 2,4%, проблем нет». В коде **две** ошибки. Найдите их, исправьте и напишите менеджеру правильную цифру.
```python
p = df.pivot_table(index="city", columns="month", values="amount")
p["chg"] = p[10] / p[9] - 1
print(p["chg"].round(3))
```

In [26]:
p = paid.pivot_table(
    index="city", columns="month", values="amount", aggfunc="sum"
)

In [27]:
p["chg"] = (p[10] / p[9] - 1) * 100

In [28]:
p["chg"].round(1)

city
Актобе       -1.4
Алматы        1.0
Астана       -2.0
Атырау        5.0
Караганда    -7.6
Шымкент     -11.3
Name: chg, dtype: float64

In [ ]:
В шымкенте выручка оплаченных заказов в октябре снизилась на 11.3% относительно сентябряФ
рост на 2.4% был получен из среднего чека по всем заказам включая отменённые, вместо суммы оплаченных заказов

## [Стандарт] 5. Возраст клиентов
Соедините заказы с клиентами (`merge`, `how="left"`, проверьте `len`). Возрастная группа: `m["age_group"] = m["age"] // 10 * 10`.
По группам: число заказов, доля отмен, средний и медианный чек оплаченных.
AI предлагает: «Группа 60 покупает дороже всех и чаще отменяет, сделаем для неё отдельную акцию». Какие возрасты попали в группу 60 и сколько в ней заказов? Подтверждает ли медиана вывод AI?

In [29]:
m = df.merge(cu, on="customer_id", how="left")

In [30]:
len(df), len(m)

(5280, 5280)

In [31]:
m["age_group"] = m["age"] // 10 * 10

In [32]:
m["age_group"].value_counts().sort_index()

age_group
10     229
20    1226
30    1241
40    1170
50    1294
60     120
Name: count, dtype: int64

In [33]:
by_age = m.groupby("age_group").agg(
    orders=("order_id", "count"),
    cancel_share=("cancel", "mean")
)

In [34]:
by_age

,orders,cancel_share
age_group,,
10,229,0.087336
20,1226,0.096248
30,1241,0.111201
40,1170,0.111111
50,1294,0.091963
60,120,0.133333


In [35]:
paid_age = m[m["status"] == "оплачен"]

In [36]:
by_age["avg_check"] = paid_age.groupby("age_group")["amount"].mean()

In [37]:
by_age.round(2)

,orders,cancel_share,avg_check
age_group,,,
10,229,0.09,14044.74
20,1226,0.10,14130.42
30,1241,0.11,15142.34
40,1170,0.11,14792.97
50,1294,0.09,14671.80
60,120,0.13,16779.13


In [38]:
m[m["age_group"] == 60]["age"].value_counts().sort_index()

age
60    120
Name: count, dtype: int64

In [39]:
by_age.loc[60]

orders            120.000000
cancel_share        0.133333
avg_check       16779.134615
Name: 60, dtype: float64

In [40]:
by_age["median_check"] = paid_age.groupby("age_group")["amount"].median()

In [41]:
by_age.round(2)

,orders,cancel_share,avg_check,median_check
age_group,,,,
10,229,0.09,14044.74,10700.0
20,1226,0.10,14130.42,10100.0
30,1241,0.11,15142.34,10670.0
40,1170,0.11,14792.97,10050.0
50,1294,0.09,14671.80,10240.0
60,120,0.13,16779.13,10495.0


In [ ]:
группа 60 лет чаще других отменяет заказы  13.33% из 120 заказов ее средний чек самый высокий но медианный чек 10495 не лидирует 
Поэтому утверждать что эта группа в целом покупает дороже всех и обосновывать этим отдельную акцию нельзя

## [Стандарт] 6. Повторные покупки
Сколько заказов у каждого клиента (`groupby("customer_id").size()`)? Сколько клиентов сделали 1, 2, 3… заказа (`value_counts().sort_index()`)? Какая доля клиентов купила больше одного раза?
Сколько клиентов заказывали и в сентябре, и в октябре (`groupby("customer_id")["month"].nunique()`)? Сколько клиентов из customers.csv не заказали ничего?

In [42]:
orders_per_client = df.groupby("customer_id").size()

In [43]:
orders_per_client

customer_id
C00001    4
C00002    1
C00003    2
C00004    2
C00005    4
         ..
C02196    1
C02197    3
C02198    1
C02199    2
C02200    1
Length: 2002, dtype: int64

In [44]:
orders_per_client.value_counts().sort_index()

1     501
2     563
3     434
4     293
5     127
6      58
7      15
8       9
9       1
10      1
Name: count, dtype: int64

In [45]:
round((orders_per_client > 1).mean() * 100, 1)

np.float64(75.0)

In [46]:
months_per_client = df.groupby("customer_id")["month"].nunique()

In [47]:
(months_per_client == 2).sum()

np.int64(1064)

In [48]:
(~cu["customer_id"].isin(df["customer_id"])).sum()

np.int64(198)

In [ ]:
повторные покупки распространены 75% клиентов сделавших заказ оформили больше одного заказа. 
в сентябре и октябре покупали 1064 клиента а 198 клиентов из базы за эти два месяца не заказывали

## [Челлендж] 7. Сколько стоил сбой доставки в Шымкенте
Допустим, в октябре в Шымкенте было бы столько же заказов и тот же средний чек оплаченных, а доля отмен осталась как в сентябре. Какой была бы выручка Шымкента в октябре? Сколько компания потеряла из-за отмен?
Сравните потерю с падением всей компании (823 890 ₸). Что было бы с выручкой компании без сбоя?

In [49]:
sh = df[df["city"] == "Шымкент"]

In [50]:
sh_months = sh.groupby("month").agg(
    orders=("order_id", "count"),
    cancel_share=("cancel", "mean")
)

In [51]:
sh_months

,orders,cancel_share
month,,
9,359,0.089136
10,417,0.306954


In [52]:
sh_paid = sh[sh["status"] == "оплачен"]

In [53]:
oct_avg = sh_paid[sh_paid["month"] == 10]["amount"].mean()

In [54]:
oct_without_failure = (
    sh_months.loc[10, "orders"]
    * (1 - sh_months.loc[9, "cancel_share"])
    * oct_avg
)

In [55]:
oct_actual = sh_paid[sh_paid["month"] == 10]["amount"].sum()

In [56]:
loss = oct_without_failure - oct_actual

In [57]:
round(oct_without_failure), oct_actual, round(loss)

(5315006, np.float64(4044010.0), 1270996)

In [58]:
round(loss / 823890 * 100, 1)

np.float64(154.3)

In [ ]:
при сентябрьской доле отмен и фактическом октябрьском числе заказов и среднем чеке выручка шымкента была бы выше примерно на 1.27
эта сумма больше общего падения компании так  как изменения в других городах частично его компенсировали
это оценка сценария а не доказанная сумма потерь именно из-за сбоя доставки

## [Челлендж] 8. Органика −6,0%: сигнал или шум?
Разбейте клиентов из customers.csv **случайно** на три группы в тех же пропорциях, что и каналы (`np.random.default_rng(7)`, `rng.choice`, `map`). Посчитайте изменение выручки оплаченных окт/сент для каждой группы. Повторите 100 раз в цикле.
В каком диапазоне меняется выручка случайной группы размером с органику (5% и 95% перцентили)? Можно ли по «органика −6,0%» делать вывод о SEO?

In [59]:
rng = np.random.default_rng(7)

In [60]:
groups = rng.choice(["A", "B", "C"], size=len(cu))

In [61]:
clients = cu[["customer_id"]].copy()

In [62]:
clients["group"] = groups

In [63]:
revenue = paid.pivot_table(
    index="customer_id", columns="month", values="amount",
    aggfunc="sum", fill_value=0
)

In [64]:
revenue = clients.join(revenue, on="customer_id").fillna(0)

In [65]:
revenue["change_pct"] = (revenue[10] / revenue[9] - 1) * 100

In [66]:
group_change = (
    revenue.groupby("group")[[9, 10]].sum()
    .assign(change_pct=lambda t: (t[10] / t[9] - 1) * 100)
)

In [67]:
group_change

,9,10,change_pct
group,,,
A,11753650.0,10461460.0,-10.993947
B,11323640.0,11423980.0,0.886111
C,12166120.0,12534080.0,3.024465


In [68]:
changes = []

In [69]:
for _ in range(100):
    revenue["random_group"] = rng.choice(["A", "B", "C"], size=len(revenue))
    totals = revenue.groupby("random_group")[[9, 10]].sum()
    changes.extend(((totals[10] / totals[9] - 1) * 100).tolist())

In [70]:
np.percentile(changes, [5, 95])

array([-11.30182258,   7.92745879])

In [ ]:
падение органики на 6% укладывается в разброс который мы получили при случайном разделении клиентов 
по этой цифре нельзя установить влияние SEO птомучто необходимы дополнительные данные о каналах и сопоставимые группы